# Bot vs Bot

Run this notebook to watch the local ResNet-A and ResNet-B policy-only checkpoints play each other.

In [ ]:
from pathlib import Path
import html
import json
import sys

import chess
import chess.svg
from IPython.display import HTML, SVG, clear_output, display

project_root = Path.cwd().resolve()
for candidate in (project_root, *project_root.parents):
    if (candidate / 'pyproject.toml').exists() and (candidate / 'src' / 'mcchess').exists():
        project_root = candidate
        break
else:
    raise RuntimeError('Could not find the McChess project root.')

src_path = project_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from mcchess.eval.arena import ArenaConfig, BotConfig, run_match  # noqa: E402

SECONDS_BETWEEN_MOVES = 4.0
NUM_GAMES = 1
MAX_PLY = 80
DEVICE = 'auto'

RESNET_A_CHECKPOINT = project_root / 'runs' / 'lichess_2026_05_2000plus_epoch20_cached_batchmetrics' / 'checkpoint.pt'
RESNET_B_CHECKPOINT = project_root / 'runs' / 'lichess_2026_05_2000plus_resnet_b_epoch20_cached_batchmetrics' / 'checkpoint.pt'
OUTPUT_PATH = project_root / 'runs' / 'eval' / 'notebook_resnet_a_vs_resnet_b.json'

missing = [path for path in (RESNET_A_CHECKPOINT, RESNET_B_CHECKPOINT) if not path.exists()]
if missing:
    raise FileNotFoundError('Missing checkpoint(s): ' + ', '.join(str(path) for path in missing))

events = []

def move_line(event):
    return '{ply:03d}. {color} {bot}: {san} ({uci})'.format(**event)

def show_board(fen, last_move_uci=None, header=''):
    board = chess.Board(fen)
    last_move = chess.Move.from_uci(last_move_uci) if last_move_uci else None
    clear_output(wait=True)
    if header:
        display(HTML('<b>' + html.escape(header) + '</b>'))
    display(SVG(chess.svg.board(board=board, lastmove=last_move, size=520)))
    if events:
        display(HTML('<pre>' + html.escape('\n'.join(move_line(event) for event in events[-24:])) + '</pre>'))

def on_move(event):
    events.append(event)
    show_board(event['fen'], event['uci'], move_line(event))

cfg = ArenaConfig(
    run_id='notebook_resnet_a_vs_resnet_b',
    output_path=str(OUTPUT_PATH),
    seed=0,
    num_games=NUM_GAMES,
    max_ply=MAX_PLY,
    move_delay_seconds=SECONDS_BETWEEN_MOVES,
    print_moves=True,
    agent=BotConfig(
        kind='policy_only',
        name='resnet_a',
        checkpoint_path=str(RESNET_A_CHECKPOINT),
        device=DEVICE,
    ),
    opponent=BotConfig(
        kind='policy_only',
        name='resnet_b',
        checkpoint_path=str(RESNET_B_CHECKPOINT),
        device=DEVICE,
    ),
)

show_board(chess.Board().fen(), header='Starting ResNet-A vs ResNet-B')
result = run_match(cfg, move_callback=on_move)

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH.write_text(json.dumps(result, indent=2, sort_keys=True) + '\n', encoding='utf-8')

final_game = result['games'][-1]
last_move = final_game['moves'][-1] if final_game['moves'] else None
summary = (
    'Done: {status}, W/D/L={wins}/{draws}/{losses}, score={score:.3f}, illegal_moves={illegal_moves}'
).format(**result)
show_board(final_game['final_fen'], last_move, summary)
display({
    'saved_result': str(OUTPUT_PATH),
    'status': result['status'],
    'wins': result['wins'],
    'draws': result['draws'],
    'losses': result['losses'],
    'score': result['score'],
    'illegal_moves': result['illegal_moves'],
})